<a href="https://colab.research.google.com/github/sourendra-optiflux/Railway-Crew-Rostering-Optimization/blob/main/rostering_v1_sheets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Optiflux_Crew_Rostering"
)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_DIR}"
    )

print("Project folder:", PROJECT_DIR)
print("\nFiles found:")

for file in sorted(PROJECT_DIR.iterdir()):
    if file.is_file():
        print(" -", file.name)


Project folder: /content/drive/MyDrive/Optiflux_Crew_Rostering

Files found:
 - crew.gsheet
 - duties.csv.gsheet


In [ ]:

%pip install -q ortools

import pandas as pd
from datetime import datetime, timedelta
from ortools.sat.python import cp_model

print("Libraries ready.")


Libraries ready.


In [ ]:

import pandas as pd

crew_path = PROJECT_DIR / "crew.csv"
duties_path = PROJECT_DIR / "duties.csv"

crew_df = pd.read_csv(crew_path, dtype=str)
duties_df = pd.read_csv(duties_path, dtype=str)

print("Crew data")
display(crew_df)

print("Duty data")
display(duties_df)


Crew data


,person_id,initial_location
0,A,Howrah
1,B,Yajpur
2,C,Howrah


Duty data


,id,start,start_location,end_location,activity,eligible
0,G1,2026-11-01 17:00,Howrah,Yajpur,Howrah Mainline,A;B
1,G2,2026-11-02 17:00,Yajpur,Howrah,Yajpur Mainline,A;B
2,G3,2026-11-02 17:00,Howrah,Yajpur,Howrah Mainline,A;C
3,G4,2026-11-03 17:00,Yajpur,Howrah,Yajpur Mainline,A;C
4,G5,2026-11-04 17:00,Howrah,Yajpur,Howrah Mainline,A;B;C


In [ ]:

required_crew_cols = {"person_id", "initial_location"}
required_duty_cols = {
    "id", "start", "start_location",
    "end_location", "activity", "eligible"
}

assert required_crew_cols.issubset(crew_df.columns), \
    f"Missing crew columns: {required_crew_cols - set(crew_df.columns)}"

assert required_duty_cols.issubset(duties_df.columns), \
    f"Missing duty columns: {required_duty_cols - set(duties_df.columns)}"

for df in [crew_df, duties_df]:
    df.columns = df.columns.str.strip()

crew_df = crew_df.fillna("")
duties_df = duties_df.fillna("")

for col in ["person_id", "initial_location"]:
    crew_df[col] = crew_df[col].str.strip()

for col in ["id", "start", "start_location",
            "end_location", "activity", "eligible"]:
    duties_df[col] = duties_df[col].str.strip()

assert (crew_df["person_id"] != "").all(), "Blank person ID found"
assert (duties_df["id"] != "").all(), "Blank duty ID found"
assert crew_df["person_id"].is_unique, "Duplicate crew IDs found"
assert duties_df["id"].is_unique, "Duplicate duty IDs found"
assert (crew_df["initial_location"] != "").all(), "Blank initial location"
assert (duties_df["start_location"] != "").all(), "Blank duty start location"
assert (duties_df["end_location"] != "").all(), "Blank duty end location"

crew = {
    row["person_id"]: {
        "initial_location": row["initial_location"]
    }
    for _, row in crew_df.iterrows()
}

people = list(crew.keys())
people_set = set(people)

duties = []

for _, row in duties_df.iterrows():
    try:
        start_dt = datetime.strptime(
            row["start"], "%Y-%m-%d %H:%M"
        )
    except ValueError:
        raise ValueError(
            f"Invalid start time for {row['id']}: {row['start']}. "
            "Use YYYY-MM-DD HH:MM."
        )

    eligible = [
        p.strip() for p in row["eligible"].split(";")
        if p.strip()
    ]

    unknown = set(eligible) - people_set
    if unknown:
        raise ValueError(
            f"Duty {row['id']} references unknown crew: {unknown}"
        )

    duties.append({
        "id": row["id"],
        "start_dt": start_dt,
        "end_dt": start_dt + timedelta(hours=12),
        "start_location": row["start_location"],
        "end_location": row["end_location"],
        "activity": row["activity"],
        "eligible": eligible,
    })

print(f"Validated {len(people)} crew members and {len(duties)} duties.")
display(pd.DataFrame([
    {
        "id": d["id"],
        "start": d["start_dt"],
        "end": d["end_dt"],
        "route": f'{d["start_location"]} -> {d["end_location"]}',
        "eligible": ";".join(d["eligible"])
    }
    for d in duties
]))


Validated 3 crew members and 5 duties.


,id,start,end,route,eligible
0,G1,2026-11-01 17:00:00,2026-11-02 05:00:00,Howrah -> Yajpur,A;B
1,G2,2026-11-02 17:00:00,2026-11-03 05:00:00,Yajpur -> Howrah,A;B
2,G3,2026-11-02 17:00:00,2026-11-03 05:00:00,Howrah -> Yajpur,A;C
3,G4,2026-11-03 17:00:00,2026-11-04 05:00:00,Yajpur -> Howrah,A;C
4,G5,2026-11-04 17:00:00,2026-11-05 05:00:00,Howrah -> Yajpur,A;B;C


In [ ]:

model = cp_model.CpModel()
n = len(duties)

# x[p, i] = 1 if person p is assigned duty i
x = {
    (p, i): model.new_bool_var(f"x_{p}_{i}")
    for p in people
    for i in range(n)
}

# Each duty is covered by at most one person.
for i in range(n):
    model.add(sum(x[p, i] for p in people) <= 1)

# Each person can start at most one duty per calendar day.
dates = sorted({d["start_dt"].date() for d in duties})

for p in people:
    for day in dates:
        indices = [
            i for i, d in enumerate(duties)
            if d["start_dt"].date() == day
        ]
        model.add(sum(x[p, i] for i in indices) <= 1)

# Ineligible assignments are forbidden.
for p in people:
    for i, d in enumerate(duties):
        if p not in d["eligible"]:
            model.add(x[p, i] == 0)

MIN_REST_HOURS = 11

# For each person, construct one route through assigned duties.
for p in people:
    arcs = []
    assigned_count = sum(x[p, i] for i in range(n))

    empty = model.new_bool_var(f"empty_{p}")
    model.add(assigned_count == 0).only_enforce_if(empty)
    model.add(assigned_count >= 1).only_enforce_if(empty.Not())
    arcs.append((0, 0, empty))

    for i, d in enumerate(duties):
        node = i + 1

        # If duty isn't assigned, its node is skipped.
        arcs.append((node, node, x[p, i].Not()))

        # A first duty must begin at the initial location.
        if d["start_location"] == crew[p]["initial_location"]:
            first = model.new_bool_var(f"first_{p}_{i}")
            arcs.append((0, node, first))

        # A selected duty can be the last duty in the route.
        last = model.new_bool_var(f"last_{p}_{i}")
        arcs.append((node, 0, last))

    for i, di in enumerate(duties):
        for j, dj in enumerate(duties):
            if i == j:
                continue

            rest_ok = (
                dj["start_dt"] >=
                di["end_dt"] + timedelta(hours=MIN_REST_HOURS)
            )
            location_ok = (
                di["end_location"] == dj["start_location"]
            )

            if rest_ok and location_ok:
                next_arc = model.new_bool_var(f"next_{p}_{i}_{j}")
                arcs.append((i + 1, j + 1, next_arc))

    model.add_circuit(arcs)

# Maximize total assignments.
model.maximize(sum(x[p, i] for p in people for i in range(n)))

print("Optimization model built.")


Optimization model built.


In [ ]:

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30

status = solver.solve(model)
print("Solver status:", solver.status_name(status))

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    covered_count = sum(
        solver.value(x[p, i])
        for p in people for i in range(n)
    )
    print(f"Covered duties: {covered_count}/{n}")
else:
    print("No feasible assignment found. Check your input data and constraints.")


Solver status: OPTIMAL
Covered duties: 5/5


In [ ]:

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    results = []
    covered_indices = set()

    for p in people:
        for i, d in enumerate(duties):
            if solver.value(x[p, i]):
                covered_indices.add(i)
                results.append({
                    "person_id": p,
                    "duty_id": d["id"],
                    "start": d["start_dt"],
                    "end": d["end_dt"],
                    "activity": d["activity"],
                    "start_location": d["start_location"],
                    "end_location": d["end_location"],
                })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(["person_id", "start"])
    display(results_df)

    uncovered_df = pd.DataFrame([
        {
            "duty_id": d["id"],
            "start": d["start_dt"],
            "activity": d["activity"],
            "start_location": d["start_location"],
            "end_location": d["end_location"],
        }
        for i, d in enumerate(duties)
        if i not in covered_indices
    ])

    print("Uncovered duties:")
    display(uncovered_df)


,person_id,duty_id,start,end,activity,start_location,end_location
0,A,G1,2026-11-01 17:00:00,2026-11-02 05:00:00,Howrah Mainline,Howrah,Yajpur
1,B,G2,2026-11-02 17:00:00,2026-11-03 05:00:00,Yajpur Mainline,Yajpur,Howrah
2,C,G3,2026-11-02 17:00:00,2026-11-03 05:00:00,Howrah Mainline,Howrah,Yajpur
3,C,G4,2026-11-03 17:00:00,2026-11-04 05:00:00,Yajpur Mainline,Yajpur,Howrah
4,C,G5,2026-11-04 17:00:00,2026-11-05 05:00:00,Howrah Mainline,Howrah,Yajpur


Uncovered duties:


""


In [ ]:

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    assignments_path = PROJECT_DIR / "assignments_output.csv"
    uncovered_path = PROJECT_DIR / "uncovered_output.csv"

    results_df.to_csv(assignments_path, index=False)
    uncovered_df.to_csv(uncovered_path, index=False)

    print("Saved assignments:", assignments_path)
    print("Saved uncovered duties:", uncovered_path)


Saved assignments: /content/drive/MyDrive/Optiflux_Crew_Rostering/assignments_output.csv
Saved uncovered duties: /content/drive/MyDrive/Optiflux_Crew_Rostering/uncovered_output.csv
